# Pipeline Tests

Notebook kiểm thử pipeline. Chạy từ thư mục gốc repo; cell đầu nạp và chạy `data_pipeline.ipynb`.

In [1]:
from pathlib import Path

pipeline_notebook = Path("notebooks/pipeline/data_pipeline.ipynb")
if not pipeline_notebook.exists():
    pipeline_notebook = Path("../pipeline/data_pipeline.ipynb")
get_ipython().run_line_magic("run", f'"{pipeline_notebook}"')

{'status': 'passed_with_warnings',
 'rows': {'products': 3341,
  'shop_info': 20,
  'category_list': 491,
  'product_categories': 4054,
  'category_platform': 4482},
 'metric_rows': {'snapshot': 3341, 'transition': 2184},
 'severity_counts': {'warning': 244},
 'report': '/Users/vqd/BA3/data/processed/pipeline_report.json'}

In [2]:
import tempfile
import unittest
from pathlib import Path

import pandas as pd

class PipelineUnitTests(unittest.TestCase):
    def test_safe_array_reports_malformed_json(self):
        parsed, error = safe_array('["a", "b"]')
        self.assertEqual(parsed, ["a", "b"])
        self.assertIsNone(error)
        parsed, error = safe_array("not-json")
        self.assertIsNone(parsed)
        self.assertIsNotNone(error)

    def test_invalid_number_is_null_and_reported(self):
        frame = pd.DataFrame([{
            "country_code": "vn", "shop_id": "1", "item_id": "2", "date": "2026-07-01",
            "product_name": "Test", "price": "not-a-number", "source_file": "fixture.csv",
            "source_row": 2, "path_country_code": "vn", "path_dataset": "products", "path_shop_id": "1",
        }])
        issues = []
        clean = normalize("products", frame, issues)
        self.assertTrue(pd.isna(clean.loc[0, "price_num"]))
        self.assertIn("INVALID_NUMBER", [issue.code for issue in issues])


class CurrentDatasetIntegrationTests(unittest.TestCase):
    def test_pipeline_reproduces_audited_contract(self):
        repo = REPO_ROOT
        with tempfile.TemporaryDirectory() as temp_dir:
            report = run_pipeline(repo / "data/raw", Path(temp_dir))
            self.assertEqual(report["status"], "passed_with_warnings")
            self.assertEqual(report["rows"]["products"], 3341)
            self.assertEqual(report["metric_rows"], {"snapshot": 3341, "transition": 2184})
            self.assertEqual(report["numeric_conversion_error_count"], 0)
            self.assertEqual(report["snapshot_checks"]["missing_cells"], 130)
            self.assertEqual(report["issue_counts"]["warning:EXACT_DUPLICATE_REMOVED"], 30)
            self.assertEqual(report["issue_counts"]["warning:INTERNAL_SNAPSHOT_GAP"], 5)
            self.assertEqual(report["issue_counts"]["warning:HISTORY_SOLD_DECREASE"], 88)
            self.assertEqual(report["issue_counts"]["warning:PRICE_SENTINEL"], 3)
            self.assertTrue(all(check["duplicate_key_rows"] == 0 for check in report["key_checks"].values()))

            transitions = pd.read_csv(Path(temp_dir) / "product_transition_metrics.csv")
            anomaly = transitions["history_sold_decrease_flag"].astype(str).str.lower().eq("true")
            self.assertEqual(int(anomaly.sum()), 88)
            self.assertTrue(transitions.loc[anomaly, "snapshot_sales_delta_clean"].isna().all())
            gap = ~transitions["transition_metric_eligible"].astype(str).str.lower().eq("true")
            self.assertTrue(transitions.loc[gap, "monthly_sold_delta"].isna().all())


In [3]:
suite = unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromTestCase(PipelineUnitTests),
    unittest.defaultTestLoader.loadTestsFromTestCase(CurrentDatasetIntegrationTests),
])
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("Pipeline notebook tests failed")

test_invalid_number_is_null_and_reported (__main__.PipelineUnitTests) ... 

ok
test_safe_array_reports_malformed_json (__main__.PipelineUnitTests) ... 

ok
test_pipeline_reproduces_audited_contract (__main__.CurrentDatasetIntegrationTests) ... 

ok

----------------------------------------------------------------------
Ran 3 tests in 2.886s

OK
